# Axolotl SFT QLoRA — Qwen3-4B & Qwen3-14B

Models not supported by Tinker RL, so we run them here via Axolotl.

**Pipeline**: SFT with QLoRA on Colab (T4/A100)

Training preferences:
- SFT LR: 3e-6 to 1e-5 (order of magnitude lower than pre-training)
- LoRA rank: 32
- Sequence packing for up to 33x token/batch efficiency
- Mask user turns (focus loss on assistant responses)
- Multiple epochs can significantly improve performance
- ~100k examples for instruction, reasoning, steerability

## 1. Environment Setup

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Install Axolotl + dependencies
!pip install -q axolotl[flash-attn] peft bitsandbytes wandb huggingface_hub
!pip install -q flash-attn --no-build-isolation

In [ ]:
# Login to HuggingFace and W&B
from huggingface_hub import login
login()  # paste your HF token

import wandb
wandb.login()  # paste your W&B API key

## 2. Experiment Configs

Qwen3-4B and Qwen3-14B — not supported by Tinker RL, so we fine-tune here.

In [ ]:
import yaml
from pathlib import Path

Path("configs").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)

### Experiment A: Qwen3-4B QLoRA SFT (T4-friendly)

In [ ]:
config_4b = {
    "base_model": "Qwen/Qwen3-4B",
    "model_type": "AutoModelForCausalLM",
    "tokenizer_type": "AutoTokenizer",

    "load_in_8bit": False,
    "load_in_4bit": True,

    "datasets": [
        {
            "path": "mlabonne/FineTome-100k",
            "type": "chat_template",
            "field_messages": "conversations",
            "message_field_role": "from",
            "message_field_content": "value",
            "roles": {"user": ["human"], "assistant": ["gpt"]},
        }
    ],
    "val_set_size": 0.02,
    "output_dir": "./outputs/qwen3-4b-sft-qlora",

    "sequence_len": 2048,
    "sample_packing": True,
    "eval_sample_packing": False,

    # Mask user turns - focus loss on assistant responses only
    "train_on_inputs": False,

    # QLoRA config
    "adapter": "qlora",
    "lora_r": 32,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "lora_target_linear": True,

    # Training
    "gradient_accumulation_steps": 8,
    "micro_batch_size": 2,
    "num_epochs": 3,
    "optimizer": "adamw_bnb_8bit",
    "lr_scheduler": "cosine",
    "learning_rate": 5e-6,  # SFT range: 3e-6 to 1e-5
    "warmup_ratio": 0.1,

    "bf16": "auto",
    "gradient_checkpointing": True,
    "flash_attention": True,

    "logging_steps": 1,
    "evals_per_epoch": 4,
    "saves_per_epoch": 1,
    "weight_decay": 0.01,

    "wandb_project": "axolotl-sft-experiments",
    "wandb_name": "qwen3-4b-qlora-lr5e6",
}

with open("configs/qwen3_4b_sft_qlora.yml", "w") as f:
    yaml.dump(config_4b, f, default_flow_style=False)

print("Wrote configs/qwen3_4b_sft_qlora.yml")
print(yaml.dump(config_4b, default_flow_style=False))

### Experiment B: Qwen3-14B QLoRA SFT (A100 recommended)

In [ ]:
config_14b = {
    "base_model": "Qwen/Qwen3-14B",
    "model_type": "AutoModelForCausalLM",
    "tokenizer_type": "AutoTokenizer",

    "load_in_8bit": False,
    "load_in_4bit": True,

    "datasets": [
        {
            "path": "mlabonne/FineTome-100k",
            "type": "chat_template",
            "field_messages": "conversations",
            "message_field_role": "from",
            "message_field_content": "value",
            "roles": {"user": ["human"], "assistant": ["gpt"]},
        }
    ],
    "val_set_size": 0.02,
    "output_dir": "./outputs/qwen3-14b-sft-qlora",

    "sequence_len": 4096,
    "sample_packing": True,
    "eval_sample_packing": False,
    "train_on_inputs": False,

    "adapter": "qlora",
    "lora_r": 32,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "lora_target_linear": True,

    "gradient_accumulation_steps": 16,
    "micro_batch_size": 1,
    "num_epochs": 3,
    "optimizer": "adamw_bnb_8bit",
    "lr_scheduler": "cosine",
    "learning_rate": 3e-6,  # Lower end for larger model
    "warmup_ratio": 0.1,

    "bf16": "auto",
    "gradient_checkpointing": True,
    "flash_attention": True,

    "logging_steps": 1,
    "evals_per_epoch": 4,
    "saves_per_epoch": 1,
    "weight_decay": 0.01,

    "wandb_project": "axolotl-sft-experiments",
    "wandb_name": "qwen3-14b-qlora-lr3e6",
}

with open("configs/qwen3_14b_sft_qlora.yml", "w") as f:
    yaml.dump(config_14b, f, default_flow_style=False)

print("Wrote configs/qwen3_14b_sft_qlora.yml")
print(yaml.dump(config_14b, default_flow_style=False))

## 3. LR Sweep Helper

Generate configs across your preferred SFT LR range (3e-6 to 1e-5) for Qwen3-4B.

In [ ]:
import copy

lr_sweep = [3e-6, 5e-6, 7e-6, 1e-5]

for lr in lr_sweep:
    cfg = copy.deepcopy(config_4b)
    cfg["learning_rate"] = lr
    cfg["wandb_name"] = f"qwen3-4b-qlora-lr{lr:.0e}"
    cfg["output_dir"] = f"./outputs/qwen3-4b-sft-lr{lr:.0e}"
    fname = f"configs/qwen3_4b_lr_{lr:.0e}.yml"
    with open(fname, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f"  {fname} -> lr={lr}")

print(f"\nGenerated {len(lr_sweep)} sweep configs")

## 4. Train

Pick a config and launch training.
- **Qwen3-4B**: fits on T4 (16GB) with QLoRA
- **Qwen3-14B**: needs A100 (40GB) with QLoRA

In [ ]:
# Choose your config
CONFIG = "configs/qwen3_4b_sft_qlora.yml"   # T4-safe
# CONFIG = "configs/qwen3_14b_sft_qlora.yml"  # A100

In [ ]:
# Preprocess dataset (tokenize + pack sequences)
!accelerate launch -m axolotl.cli.preprocess {CONFIG}

In [ ]:
# Launch training
!accelerate launch -m axolotl.cli.train {CONFIG}

## 5. Evaluate & Inference

In [ ]:
# Interactive inference with the trained model
!accelerate launch -m axolotl.cli.inference {CONFIG} \
    --lora-model-dir ./outputs/qwen3-4b-sft-qlora

In [ ]:
# Quick test with transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model = "Qwen/Qwen3-4B"
adapter_path = "./outputs/qwen3-4b-sft-qlora"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model, torch_dtype=torch.bfloat16, device_map="auto"
)
model = PeftModel.from_pretrained(model, adapter_path)

messages = [
    {"role": "user", "content": "Explain the difference between SFT and DPO in 3 sentences."}
]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = model.generate(inputs, max_new_tokens=256, temperature=0.7, do_sample=True)

print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 6. Merge & Push to Hub

In [ ]:
# Merge LoRA weights into base model
!accelerate launch -m axolotl.cli.merge_lora {CONFIG} \
    --lora-model-dir ./outputs/qwen3-4b-sft-qlora \
    --output-dir ./outputs/qwen3-4b-sft-merged

In [ ]:
# Push merged model to HuggingFace Hub
from huggingface_hub import HfApi

api = HfApi()
# api.upload_folder(
#     folder_path="./outputs/qwen3-4b-sft-merged",
#     repo_id="YOUR_USERNAME/qwen3-4b-sft",
#     repo_type="model",
# )